## ÉTAPE 3 — Statistiques avec groupby()
Objectif : top 10 des cultures les plus produites


In [ ]:
import pandas as pd


## 1. CHARGER ET PRÉPARER LES DONNÉES


In [ ]:
df = pd.read_csv("JEU_DE_DONNEE.csv")
print(f"Dataset chargé : {df.shape[0]:,} lignes\n")

# On garde uniquement la production mondiale et la quantité produite
# "World +" = données agrégées pour le monde entier
df_production = df[
    (df["country_or_area"] == "World +") &
    (df["element"] == "Production Quantity")
].copy()

# Supprimer les lignes sans valeur (NaN)
df_production = df_production.dropna(subset=["value"])

print(f"Lignes après filtrage : {df_production.shape[0]:,}\n")


## 2. GROUPBY — ADDITIONNER PAR CULTURE


In [ ]:
# groupby("category") regroupe toutes les lignes d'une même culture
# .sum() additionne les valeurs de chaque groupe
# Le résultat : une ligne par culture avec le total sur toutes les années

production_par_culture = df_production.groupby("category")["value"].sum()

print("--- Exemple : quelques cultures et leur total ---")
print(production_par_culture.head(10))
print()


## 3. TRIER ET GARDER LE TOP 10


In [ ]:
# .sort_values(ascending=False) trie du plus grand au plus petit
# .head(10) garde seulement les 10 premières lignes
top10 = production_par_culture.sort_values(ascending=False).head(10)

print("--- TOP 10 des cultures les plus produites (toutes années) ---")
print(top10)
print()


## 4. AFFICHAGE PLUS LISIBLE AVEC RESET_INDEX


In [ ]:
# .reset_index() transforme le résultat en DataFrame propre
# avec deux colonnes : category et value
top10_df = (
    df_production
    .groupby("category")["value"]
    .sum()
    .sort_values(ascending=False)
    .head(10)
    .reset_index()
)

# Renommer les colonnes pour plus de clarté
top10_df.columns = ["Culture", "Production totale"]

# Ajouter un rang (1er, 2ème, etc.)
top10_df.index = top10_df.index + 1  # commence à 1 au lieu de 0
top10_df.index.name = "Rang"

print("--- TOP 10 formaté ---")
print(top10_df.to_string())
print()


## 5. STATISTIQUES SUPPLÉMENTAIRES


In [ ]:
# Nombre total de cultures différentes
nb_cultures = df_production["category"].nunique()
print(f"Nombre total de cultures dans le dataset : {nb_cultures}")

# Part du top 10 dans la production mondiale totale
total_mondial = df_production["value"].sum()
total_top10 = top10_df["Production totale"].sum()
part = (total_top10 / total_mondial) * 100

print(f"Production mondiale totale  : {total_mondial:,.0f}")
print(f"Production du top 10        : {total_top10:,.0f}")
print(f"Part du top 10              : {part:.1f}% de la production mondiale")
print()


## 6. ALLER PLUS LOIN — GROUPBY SUR PLUSIEURS COLONNES


In [ ]:
# On peut aussi grouper par culture ET par année
# Utile pour voir l'évolution dans le temps
production_par_annee = (
    df_production
    .groupby(["category", "year"])["value"]
    .sum()
    .reset_index()
)

print("--- Production par culture ET par année (extrait) ---")
print(production_par_annee[production_par_annee["category"] == "wheat"].head(10))
print()


## 7. SAUVEGARDER LE TOP 10


In [ ]:
top10_df.to_csv("top10_cultures.csv")
print("Fichier 'top10_cultures.csv' sauvegardé ✓")
